In [26]:
import openai

client = openai.OpenAI()

PROMPT = """
당신은 영화를 추천하는 챗봇입니다. 
사용자가 좋아하는 장르를 기억해야하고, 이미 시청한 영화를 기억해야합니다.
그리고 사용자와의 대화 내용을 기반으로 개인화된 추천을 제공할 수 있어야 합니다. 
"""
messages = [{"role": "system", "content": PROMPT}]

def get_ai_response():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")


In [27]:
import requests

base_url = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    endpoint = "/movies"
    url = base_url + endpoint
    response = requests.get(url)

    if response.status_code == 200:
        return(response.json())

def get_moive_details(id):
    endpoint = f"/movies/{id}"
    url = base_url + endpoint
    response = requests.get(url)

    if response.status_code == 200:
        return(response.json())



def get_similar_movies(id):
    endpoint = f"/movies/{id}/similar"
    url = base_url + endpoint
    response = requests.get(url)

    if response.status_code == 200:
        return(response.json())


FUNCTION_MAP = {
    "get_popular_movies" : get_popular_movies,
    "get_movie_details" : get_moive_details,
    "get_similar_moveis" : get_similar_movies,
}


In [28]:
TOOLS:[
    {
        "type": "function",
        "function":{
            "name": "get_popular_movies()",
            "description": "Send a request to https://nomad-movies-2.nomadcoders.workers.dev/movies and get a list of popular movies",
        }
    },
    {
        "type": "function",
        "function":{
            "name":"get_movie_details",
            "description": "return details of a movie in https://nomad-movies-2.nomadcoders.workers.dev/movies/:id",
            "parameters":{
                "type":"object",
                "properties":{
                    "movie_id":{
                        "type":"string",
                        "description": "The ID of the movie to get the details of.",
                    }
                },
                "required":["movie_id"],
            }
        }
    },
    {
        "type": "function",
        "function":{
            "name":"get_similar_movies",
            "description": "return details of a movie in https://nomad-movies-2.nomadcoders.workers.dev/movies/:id/similar",
            "parameters":{
                "type":"object",
                "properties":{
                    "movie_id":{
                        "type":"string",
                        "description": "The ID of the movie to get the similar movies of.",
                    }
                },
                "required":["movie_id"],
            }
        }
    }
]

In [29]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        get_ai_response()
        
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")



In [30]:
while True:
    message = input("Movie Agent에게 무엇이든 물어보세요!")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        get_ai_response()

User: 지금 인기 있는 영화 알려줘
AI: 현재 인기 있는 영화들 중 일부는 다음과 같습니다:

1. **오픈하이머 (Oppenheimer)** - 크리스토퍼 놀란 감독의 전기 드라마로, 핵무기를 개발한 로버트 오펜하이머의 삶을 다룹니다.
2. **바비 (Barbie)** - 그레타 거윅 감독의 코미디 영화로, 바비 인형을 주인공으로 한 이야기를 담고 있습니다.
3. **마션 (The Martian)** - 우주에서 고립된 한 남자가 생존을 위해 고군분투하는 이야기를 그린 SF 영화입니다.
4. **원피스 필름 레드 (One Piece Film: Red)** - 인기 애니메이션 원피스를 바탕으로 한 최신 영화로, 팬들에게 큰 인기를 끌고 있습니다.

더 알고 싶은 특정 장르가 있으신가요? 또는 보고 싶었던 영화가 있으신가요?
User: 오픈하이머 영화에 대해 더 알려줘
AI: **오픈하이머 (Oppenheimer)**는 2023년에 개봉한 크리스토퍼 놀란 감독의 전기 드라마 영화로, 로버트 오펜하이머(Robert Oppenheimer)의 이야기를 중심으로 전개됩니다. 그는 제2차 세계 대전 동안 미국의 맨해튼 프로젝트를 이끌며 최초의 원자폭탄 개발에 기여한 물리학자입니다.

**주요 내용:**
- 영화는 오펜하이머의 생애와 그가 겪는 윤리적 갈등, 전쟁의 비극적 결과를 조명합니다. 
- 그의 천재성, 동료 과학자들과의 관계, 그리고 결국 폭탄의 사용이 가져온 인류에 대한 영향에 대한 깊은 질문들을 제기합니다.
- 영화는 과거의 사건을 다시 조명하는 방식으로 진행되며, 시청자들에게 과학의 발전과 그로 인한 도덕적 책임에 대한 고민을 불러일으킵니다.

영화는 배우들이 대거 참여하며, 특히 주연인 킬리언 머피가 오펜하이머 역을 맡아 뛰어난 연기를 선보이고 있습니다. 비주얼 효과와 미술, 음악 등에서도 높은 평가를 받고 있는 작품입니다.

이 영화에 대해 더 궁금한 점이 있으신가요? 또는 이와 비슷한 다른 영화를 추천해드릴까요?
User: 비슷한 영화 추천해줘
A